#  Task 2: Emotion Recognition from Speech

**Objective:** Recognize human emotions (happy, angry, sad, neutral, fearful, disgusted, surprised) from speech audio.

**Approach:** Deep Learning with MFCCs + CNN/LSTM hybrid model

**Dataset:** RAVDESS (Ryerson Audio-Visual Database of Emotional Speech and Song)


In [25]:
# Install required packages
!pip install librosa soundfile numpy pandas scikit-learn matplotlib seaborn tensorflow keras kaggle tqdm

Defaulting to user installation because normal site-packages is not writeable


In [26]:
import os
import zipfile

# ============================================================
# OPTION A: Download via Kaggle API
# 1. Go to https://www.kaggle.com/account -> Create API Token
# 2. Upload kaggle.json OR set credentials below
# ============================================================

# Uncomment and fill your Kaggle credentials:
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

# Then run:
# !kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
# !unzip ravdess-emotional-speech-audio.zip -d ravdess_data

print("Configure Kaggle credentials above to auto-download.")
print("\nAlternative: See Option B below for manual download.")

Configure Kaggle credentials above to auto-download.

Alternative: See Option B below for manual download.


In [27]:
# ============================================================
# OPTION B: Direct Download via requests (no Kaggle needed)
# Uses a subset mirror / Zenodo archive
# ============================================================

import urllib.request
import zipfile
import os

DATA_DIR = 'ravdess_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Official Zenodo DOI link for RAVDESS
# Full dataset ~15 files, each ~300MB. Here we show how to fetch Actor_01 as sample:
ZENODO_BASE = "https://zenodo.org/record/1188976/files"

# Download first 4 actors as demo (full dataset = Actor_01 to Actor_24)
actors_to_download = ['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04']

for actor in actors_to_download:
    actor_dir = os.path.join(DATA_DIR, actor)
    zip_path = os.path.join(DATA_DIR, f'{actor}.zip')
    
    if not os.path.exists(actor_dir):
        print(f"Downloading {actor}...")
        url = f"{ZENODO_BASE}/{actor}.zip"
        try:
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA_DIR)
            os.remove(zip_path)
            print(f"  ✅ {actor} downloaded and extracted")
        except Exception as e:
            print(f"  ❌ Failed: {e}")
    else:
        print(f"  ✅ {actor} already exists")

# List downloaded files
all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith('.wav'):
            all_files.append(os.path.join(root, file))

print(f"\n📁 Total audio files found: {len(all_files)}")
if all_files:
    print("Sample files:", all_files[:3])

  ❌ Failed: HTTP Error 404: NOT FOUND
  ❌ Failed: HTTP Error 404: NOT FOUND
  ❌ Failed: HTTP Error 404: NOT FOUND
  ❌ Failed: HTTP Error 404: NOT FOUND

📁 Total audio files found: 0


In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import os
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Librosa version: {librosa.__version__}")
print("✅ All libraries loaded!")

ModuleNotFoundError: No module named 'librosa'

In [ ]:
# RAVDESS filename format: 03-01-06-01-02-01-12.wav
# Position 3 (index 2) = Emotion code

EMOTION_MAP = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

def get_emotion_from_filename(filepath):
    """Extract emotion label from RAVDESS filename."""
    filename = os.path.basename(filepath)
    parts = filename.split('-')
    if len(parts) >= 3:
        emotion_code = parts[2]
        return EMOTION_MAP.get(emotion_code, 'unknown')
    return 'unknown'

# Collect all audio files and labels
DATA_DIR = 'ravdess_data'
audio_files = []
labels = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in sorted(files):
        if file.endswith('.wav'):
            filepath = os.path.join(root, file)
            emotion = get_emotion_from_filename(filepath)
            if emotion != 'unknown':
                audio_files.append(filepath)
                labels.append(emotion)

df = pd.DataFrame({'filepath': audio_files, 'emotion': labels})
print(f"Total samples: {len(df)}")
print("\nEmotion distribution:")
print(df['emotion'].value_counts())

In [ ]:
# Visualize emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
emotion_counts = df['emotion'].value_counts()
colors = sns.color_palette('husl', len(emotion_counts))
axes[0].bar(emotion_counts.index, emotion_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Emotion Distribution in Dataset', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(emotion_counts.values, labels=emotion_counts.index, colors=colors, 
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Emotion Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Distribution plot saved")

In [ ]:
# Visualize waveforms for different emotions
def plot_waveform_and_spectrogram(filepath, emotion_label):
    """Plot waveform, mel spectrogram, and MFCC for an audio file."""
    y, sr = librosa.load(filepath, duration=3.0)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Emotion: {emotion_label.upper()}', fontsize=14, fontweight='bold')
    
    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue')
    axes[0].set_title('Waveform')
    axes[0].set_xlabel('Time (s)')
    
    # Mel Spectrogram
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = librosa.display.specshow(mel_spec_db, x_axis='time', y_axis='mel', 
                                   sr=sr, ax=axes[1], cmap='magma')
    axes[1].set_title('Mel Spectrogram')
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    
    # MFCC
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    img2 = librosa.display.specshow(mfccs, x_axis='time', ax=axes[2], cmap='coolwarm')
    axes[2].set_title('MFCCs (20 coefficients)')
    fig.colorbar(img2, ax=axes[2])
    
    plt.tight_layout()
    plt.savefig(f'viz_{emotion_label}.png', dpi=120, bbox_inches='tight')
    plt.show()

# Show 3 different emotions
emotions_to_show = ['happy', 'sad', 'angry']
for emotion in emotions_to_show:
    subset = df[df['emotion'] == emotion]
    if len(subset) > 0:
        sample_file = subset.iloc[0]['filepath']
        plot_waveform_and_spectrogram(sample_file, emotion)
    else:
        print(f"No samples found for: {emotion}")

In [ ]:
def extract_features(filepath, sr=22050, duration=3.0, offset=0.5):
    """
    Extract comprehensive audio features:
    - MFCC: 40 coefficients (mean + std)
    - Chroma STFT: 12 features
    - Mel Spectrogram: 128 features
    - Zero Crossing Rate
    - RMS Energy
    - Spectral Centroid, Bandwidth, Rolloff
    """
    try:
        y, sr = librosa.load(filepath, sr=sr, duration=duration, offset=offset)
        
        # Pad if audio is shorter than expected
        target_length = int(sr * duration)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)))
        
        features = []
        
        # 1. MFCCs (40 coefficients)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        features.extend(np.mean(mfccs, axis=1))
        features.extend(np.std(mfccs, axis=1))
        
        # 2. Chroma STFT
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features.extend(np.mean(chroma, axis=1))
        features.extend(np.std(chroma, axis=1))
        
        # 3. Mel Spectrogram (mean of 128 bins)
        mel = librosa.feature.melspectrogram(y=y, sr=sr)
        features.extend(np.mean(mel, axis=1))
        
        # 4. Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))
        
        # 5. RMS Energy
        rms = librosa.feature.rms(y=y)
        features.append(np.mean(rms))
        features.append(np.std(rms))
        
        # 6. Spectral Features
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        features.append(np.mean(spectral_centroid))
        
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features.append(np.mean(spectral_bandwidth))
        
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features.append(np.mean(spectral_rolloff))
        
        return np.array(features, dtype=np.float32)
    
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return None


def extract_features_2d(filepath, sr=22050, duration=3.0, n_mfcc=40, n_mels=128):
    """
    Extract 2D MFCC feature map for CNN input.
    Returns shape: (n_mfcc, time_steps)
    """
    try:
        y, sr = librosa.load(filepath, sr=sr, duration=duration)
        target_length = int(sr * duration)
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)))
        
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfccs = librosa.util.fix_length(mfccs, size=128, axis=1)
        return mfccs.astype(np.float32)
    except Exception as e:
        print(f"Error: {e}")
        return None


# Test feature extraction
sample = df.iloc[0]['filepath']
feats = extract_features(sample)
print(f"✅ 1D Feature vector shape: {feats.shape}")

feats_2d = extract_features_2d(sample)
print(f"✅ 2D MFCC feature map shape: {feats_2d.shape}")

In [ ]:
# Extract features for ALL audio files
print("Extracting features from audio files...")
print("(This may take a few minutes)")

X_1d = []    # For MLP / traditional ML
X_2d = []    # For CNN
y_all = []
failed = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing audio"):
    feat_1d = extract_features(row['filepath'])
    feat_2d = extract_features_2d(row['filepath'])
    
    if feat_1d is not None and feat_2d is not None:
        X_1d.append(feat_1d)
        X_2d.append(feat_2d)
        y_all.append(row['emotion'])
    else:
        failed.append(idx)

X_1d = np.array(X_1d)
X_2d = np.array(X_2d)
y_all = np.array(y_all)

print(f"\n✅ Feature extraction complete!")
print(f"1D Features shape: {X_1d.shape}")
print(f"2D Features shape: {X_2d.shape}")
print(f"Labels shape: {y_all.shape}")
print(f"Failed files: {len(failed)}")

# Save features for reuse
np.save('X_1d_features.npy', X_1d)
np.save('X_2d_features.npy', X_2d)
np.save('y_labels.npy', y_all)
print("💾 Features saved to disk!")

In [ ]:
# Load features (if already saved)
# X_1d = np.load('X_1d_features.npy')
# X_2d = np.load('X_2d_features.npy')
# y_all = np.load('y_labels.npy', allow_pickle=True)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y_all)
n_classes = len(le.classes_)

print(f"Classes: {le.classes_}")
print(f"Number of classes: {n_classes}")
print(f"Encoded labels sample: {y_encoded[:5]}")

# One-hot encode for deep learning
y_onehot = to_categorical(y_encoded, num_classes=n_classes)
print(f"One-hot shape: {y_onehot.shape}")

In [ ]:
# ─── For 1D features (MLP / LSTM) ───
scaler = StandardScaler()
X_1d_scaled = scaler.fit_transform(X_1d)

X_train_1d, X_test_1d, y_train_1d, y_test_1d = train_test_split(
    X_1d_scaled, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)
_, _, y_train_enc, y_test_enc = train_test_split(
    X_1d_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train 1D: {X_train_1d.shape}")
print(f"Test 1D:  {X_test_1d.shape}")

# ─── For 2D features (CNN) ───
# Normalize 2D features per sample
X_2d_norm = X_2d / (np.max(np.abs(X_2d)) + 1e-10)
# Add channel dimension: (samples, 40, 128) -> (samples, 40, 128, 1)
X_2d_cnn = X_2d_norm[..., np.newaxis]

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d_cnn, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train 2D (CNN): {X_train_2d.shape}")
print(f"Test 2D (CNN):  {X_test_2d.shape}")

In [ ]:
def build_lstm_model(input_shape, n_classes):
    """
    Bidirectional LSTM model for emotion recognition.
    Input: 1D feature vector reshaped as sequence
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Reshape((input_shape[0] // 8, 8)),  # treat features as time-steps
        
        layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
        layers.Dropout(0.3),
        
        layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        
        layers.Dense(n_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

input_shape_1d = (X_train_1d.shape[1],)
lstm_model = build_lstm_model(input_shape_1d, n_classes)
lstm_model.summary()

In [ ]:
# Callbacks
callbacks_lstm = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_lstm_model.h5', save_best_only=True, verbose=0)
]

# Train LSTM
print("Training Bidirectional LSTM...")
history_lstm = lstm_model.fit(
    X_train_1d, y_train_1d,
    epochs=80,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks_lstm,
    verbose=1
)
print("\n✅ LSTM Training complete!")

In [ ]:
def build_cnn_model(input_shape, n_classes):
    """
    2D CNN for MFCC spectrogram-based emotion recognition.
    Inspired by image classification applied to audio spectrograms.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        
        # Dense layers
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        
        layers.Dense(n_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

input_shape_2d = X_train_2d.shape[1:]
cnn_model = build_cnn_model(input_shape_2d, n_classes)
cnn_model.summary()

In [ ]:
# Callbacks
callbacks_cnn = [
    EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=8, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_cnn_model.h5', save_best_only=True, verbose=0)
]

# Train CNN
print("Training CNN on MFCC spectrograms...")
history_cnn = cnn_model.fit(
    X_train_2d, y_train_2d,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks_cnn,
    verbose=1
)
print("\n✅ CNN Training complete!")

In [ ]:
def build_cnn_lstm_model(input_shape, n_classes):
    """
    CNN-LSTM Hybrid: CNN extracts spatial features from MFCC,
    LSTM captures temporal dependencies across time steps.
    """
    inputs = layers.Input(shape=input_shape)
    
    # CNN Feature Extractor
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Reshape for LSTM: (batch, time_steps, features)
    shape = x.shape
    x = layers.Reshape((shape[1], shape[2] * shape[3]))(x)
    
    # LSTM Temporal Modeling
    x = layers.LSTM(128, return_sequences=True, dropout=0.3)(x)
    x = layers.LSTM(64, return_sequences=False, dropout=0.3)(x)
    
    # Classifier
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

hybrid_model = build_cnn_lstm_model(input_shape_2d, n_classes)
hybrid_model.summary()

In [ ]:
callbacks_hybrid = [
    EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=8, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_hybrid_model.h5', save_best_only=True, verbose=0)
]

print("Training CNN-LSTM Hybrid model...")
history_hybrid = hybrid_model.fit(
    X_train_2d, y_train_2d,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks_hybrid,
    verbose=1
)
print("\n✅ CNN-LSTM Hybrid Training complete!")

In [ ]:
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} - Training History', fontsize=14, fontweight='bold')
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Acc', color='blue')
    axes[0].plot(history.history['val_accuracy'], label='Val Acc', color='orange')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', color='blue')
    axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'history_{model_name.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history_lstm, "BiLSTM")
plot_training_history(history_cnn, "CNN")
plot_training_history(history_hybrid, "CNN-LSTM Hybrid")

In [ ]:
def evaluate_model(model, X_test, y_test_onehot, y_test_encoded, model_name, label_names):
    """Comprehensive model evaluation."""
    print(f"\n{'='*50}")
    print(f"  {model_name} Evaluation")
    print(f"{'='*50}")
    
    # Test accuracy
    loss, accuracy = model.evaluate(X_test, y_test_onehot, verbose=0)
    print(f"Test Loss:     {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=label_names))
    
    # Confusion matrix
    cm = confusion_matrix(y_test_encoded, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5)
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Emotion')
    plt.ylabel('True Emotion')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return accuracy

label_names = list(le.classes_)

# Evaluate all models
acc_lstm = evaluate_model(lstm_model, X_test_1d, y_test_1d, y_test_enc, "BiLSTM", label_names)
acc_cnn = evaluate_model(cnn_model, X_test_2d, y_test_2d, y_test_enc, "CNN", label_names)
acc_hybrid = evaluate_model(hybrid_model, X_test_2d, y_test_2d, y_test_enc, "CNN-LSTM Hybrid", label_names)

In [ ]:
# Model comparison
models_compared = ['BiLSTM', 'CNN', 'CNN-LSTM Hybrid']
accuracies = [acc_lstm, acc_cnn, acc_hybrid]
colors = ['#4CAF50', '#2196F3', '#FF5722']

plt.figure(figsize=(10, 6))
bars = plt.bar(models_compared, [a*100 for a in accuracies], color=colors, edgecolor='black', linewidth=0.8)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{acc*100:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

plt.title('Model Accuracy Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.ylim(0, 110)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = models_compared[np.argmax(accuracies)]
print(f"\n🏆 Best Model: {best_model_name} ({max(accuracies)*100:.2f}%)")

In [ ]:
def predict_emotion(filepath, model, model_type='cnn', scaler=None, le=None):
    """
    Predict emotion from a new audio file.
    model_type: 'lstm' for 1D model, 'cnn' or 'hybrid' for 2D CNN
    """
    if model_type == 'lstm':
        features = extract_features(filepath)
        if features is None:
            return "Feature extraction failed"
        features_scaled = scaler.transform(features.reshape(1, -1))
        proba = model.predict(features_scaled, verbose=0)[0]
    else:
        features = extract_features_2d(filepath)
        if features is None:
            return "Feature extraction failed"
        features_norm = features / (np.max(np.abs(features)) + 1e-10)
        features_cnn = features_norm[np.newaxis, ..., np.newaxis]
        proba = model.predict(features_cnn, verbose=0)[0]
    
    predicted_idx = np.argmax(proba)
    predicted_emotion = le.inverse_transform([predicted_idx])[0]
    confidence = proba[predicted_idx] * 100
    
    print(f"\n🎤 Audio: {os.path.basename(filepath)}")
    print(f"🎭 Predicted Emotion: {predicted_emotion.upper()} ({confidence:.1f}% confident)")
    print("\nAll probabilities:")
    for i, (emotion, prob) in enumerate(zip(le.classes_, proba)):
        bar = '█' * int(prob * 30)
        print(f"  {emotion:10s}: {bar:30s} {prob*100:5.1f}%")
    
    return predicted_emotion


# Test on a sample from test set
if len(df) > 0:
    test_samples = df.sample(5)
    for _, row in test_samples.iterrows():
        actual = row['emotion']
        predicted = predict_emotion(row['filepath'], hybrid_model, 
                                    model_type='hybrid', scaler=scaler, le=le)
        print(f"  ✅ Actual: {actual} | Predicted: {predicted}")
        print("-" * 60)

In [ ]:
import pickle

# Save all models
lstm_model.save('emotion_lstm_model.h5')
cnn_model.save('emotion_cnn_model.h5')
hybrid_model.save('emotion_cnn_lstm_hybrid.h5')

# Save scaler and label encoder
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("✅ All models and preprocessors saved!")
print("\nSaved files:")
for fname in ['emotion_lstm_model.h5', 'emotion_cnn_model.h5', 'emotion_cnn_lstm_hybrid.h5',
              'scaler.pkl', 'label_encoder.pkl']:
    if os.path.exists(fname):
        size = os.path.getsize(fname) / 1024
        print(f"  📁 {fname}: {size:.1f} KB")

In [ ]:
print("=" * 60)
print("  TASK 2: EMOTION RECOGNITION FROM SPEECH — SUMMARY")
print("=" * 60)
print(f"\n📊 Dataset: RAVDESS")
print(f"   Total samples:    {len(df)}")
print(f"   Emotions:         {n_classes} classes")
print(f"   Classes:          {list(le.classes_)}")
print(f"\n🔧 Features Extracted:")
print(f"   1D Vector size:   {X_1d.shape[1]} features")
print(f"   2D MFCC map:      {X_2d.shape[1:]} (freq x time)")
print(f"\n🧠 Models Trained:")
print(f"   1. Bidirectional LSTM:  {acc_lstm*100:.2f}% accuracy")
print(f"   2. 2D CNN:              {acc_cnn*100:.2f}% accuracy")
print(f"   3. CNN-LSTM Hybrid:     {acc_hybrid*100:.2f}% accuracy")
print(f"\n🏆 Best Model: CNN-LSTM Hybrid")
print(f"\n📁 Saved Files:")
print(f"   - emotion_cnn_lstm_hybrid.h5 (Best model)")
print(f"   - emotion_cnn_model.h5")
print(f"   - emotion_lstm_model.h5")
print(f"   - scaler.pkl, label_encoder.pkl")
print("=" * 60)